In [20]:
from dotenv import load_dotenv
import os
import sqlite3
import pandas as pd
from pathlib import Path
from groq import Groq

In [21]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if GROQ_API_KEY:
    print("Groq API key loaded successfully")
else:
    print("Groq API key not found")

Groq API key loaded successfully


In [22]:
Path.cwd()

WindowsPath('d:/Office_Project/Chatbot')

In [23]:
BASE_DIR = Path.cwd()

DATA_DIR = BASE_DIR / "data"
DB_DIR = BASE_DIR / "database"
DB_PATH = DB_DIR / "sales_poc.db"

DB_DIR.mkdir(exist_ok=True)

print("Base directory:", BASE_DIR)
print("Data directory:", DATA_DIR)
print("Database path:", DB_PATH)

Base directory: d:\Office_Project\Chatbot
Data directory: d:\Office_Project\Chatbot\data
Database path: d:\Office_Project\Chatbot\database\sales_poc.db


In [24]:
conn = sqlite3.connect(DB_PATH)

cursor = conn.cursor()

print("SQLite database connected successfully")

SQLite database connected successfully


In [25]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    city TEXT,
    email TEXT
)
""")

print("customers table created")

customers table created


In [26]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT,
    price REAL
)
""")

print("products table created")

products table created


In [27]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    payment_method TEXT,
    FOREIGN KEY(customer_id)
        REFERENCES customers(customer_id)
)
""")

print("orders table created")

orders table created


In [28]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    unit_price REAL,
    total_price REAL,
    FOREIGN KEY(order_id)
        REFERENCES orders(order_id),
    FOREIGN KEY(product_id)
        REFERENCES products(product_id)
)
""")

print("order_items table created")

order_items table created


In [29]:
conn.commit()

print("All tables committed successfully")

All tables committed successfully


In [30]:
tables_df = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)

tables_df

,name
0,customers
1,products
2,orders
3,order_items


In [31]:
for table_name in ["orders", "order_items"]:
    print(f"\nForeign keys for table: {table_name}")
    
    fk_df = pd.read_sql_query(
        f"PRAGMA foreign_key_list({table_name})",
        conn
    )
    
    display(fk_df)


Foreign keys for table: orders


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,customers,customer_id,customer_id,NO ACTION,NO ACTION,NONE



Foreign keys for table: order_items


,id,seq,table,from,to,on_update,on_delete,match
0,0,0,products,product_id,product_id,NO ACTION,NO ACTION,NONE
1,1,0,orders,order_id,order_id,NO ACTION,NO ACTION,NONE


In [32]:
cursor.execute("PRAGMA foreign_keys = ON")

fk_status = cursor.execute("PRAGMA foreign_keys").fetchone()[0]

print("Foreign key enforcement:", fk_status)

Foreign key enforcement: 1


In [33]:
csv_table_mapping = {
    DATA_DIR / "customers.csv": "customers",
    DATA_DIR / "products.csv": "products",
    DATA_DIR / "orders.csv": "orders",
    DATA_DIR / "order_items.csv": "order_items",
}

In [34]:
csv_table_mapping

{WindowsPath('d:/Office_Project/Chatbot/data/customers.csv'): 'customers',
 WindowsPath('d:/Office_Project/Chatbot/data/products.csv'): 'products',
 WindowsPath('d:/Office_Project/Chatbot/data/orders.csv'): 'orders',
 WindowsPath('d:/Office_Project/Chatbot/data/order_items.csv'): 'order_items'}

In [35]:
for csv_file, table_name in csv_table_mapping.items():
    df = pd.read_csv(csv_file)
    
    df.to_sql(
        table_name,
        conn,
        if_exists="append",
        index=False
    )
    
    print(f"{csv_file.name} inserted into {table_name}")

customers.csv inserted into customers
products.csv inserted into products
orders.csv inserted into orders
order_items.csv inserted into order_items


In [36]:
conn.commit()

print("CSV data inserted successfully")

CSV data inserted successfully


In [37]:
table_names = ["customers", "products", "orders", "order_items"]

for table_name in table_names:
    count = cursor.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"{table_name}: {count} rows")

customers: 20 rows
products: 15 rows
orders: 40 rows
order_items: 80 rows


In [38]:
query = """
SELECT
    p.product_name,
    SUM(oi.total_price) AS total_sales
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.product_name
ORDER BY total_sales DESC
LIMIT 5
"""

top_products_df = pd.read_sql_query(query, conn)

top_products_df

,product_name,total_sales
0,Laptop_1,1711361.0
1,Laptop_4,736542.0
2,Monitor_13,505920.0
3,Mouse_9,460044.0
4,Monitor_15,444200.0
